# Assignment 2. Data Acquirance and Statistical Tests

*Foundations of Data Science*  
*Dr. Khalaj (Fall 2024)*  



### Description
In the first part of this homework, you are going to get familiar with Python tools used for web scraping and data crawling. Next, you will thoroughly investigate the tools and methods frequently used in statistics.

### Information  
Complete the information box below.

In [1]:
full_name = "Rouzbeh Pourjafarian"
student_id = "400106279"

### Import necessary packages

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as ss
import re
import string
from bs4 import BeautifulSoup 
import sqlite3
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import defaultdict
import time
# ====================================
# feel free to import any other package
# ====================================

## 1. NBA Leaders!

The <b><a href="https://global.nba.com/"> NBA official website</a></b> offers the latest news on players, teams, and playoffs for basketball fans. For this task, we'll focus solely on players from the 2024-2025 season. Our plan is to scrape the freely available data from the site and then analyze it statistically.

### 1.1 Crawling Season Leaders Data

Inspect the webpage at https://global.nba.com/statistics/

It contains the top 50 season leaders along with their statistics. The columns in the table represent the following :

* RANK: The player's current ranking
* PLAYER: The player's name
* TEAM: The player's current team
* G: Games played
* GS: Games started
* PPG: Points per game
* RPG: Rebounds per game
* APG: Assists per game
* MPG: Minutes played per game
* EFF: Efficiency rating, a measure of overall statistical contribution
* FG%: Field goal percentage (how often a player makes a shot)
* 3P%: 3-point field goal percentage
* FT%: Free throw percentage
* OFF: Offensive rebounds per game
* DEF: Defensive rebounds per game
* SPG: Steals per game
* BPG: Blocks per game
* TO: Turnovers per game
* PF: Personal fouls per game
* TO: Turnovers per game
* PF: Personal fouls per game
* PO: Points per game

Using the `BeautifulSoup` package, scrape the data from this webpage. You must first scrape and save the data to a Python dictionary. To store the data, you will be using a SQLite database.

In [ ]:
def scrape_nba_data(url):
    data = []
    dict_data = defaultdict(list)
    driver = webdriver.Edge()
    try:
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "nba-stat-table__overflow"))
        )

        # Locate the table and extract headers and rows
        table_parent = driver.find_element(By.CLASS_NAME, "nba-stat-table__overflow")
        table = table_parent.find_element(By.XPATH, "*")
        rows = table.find_elements(By.XPATH, '//tbody/tr')

        # Iterate over each row and extract player data
        stat_names = "G	GS	PPG	RPG	APG	MPG	EFF	FG%	3P%	FT%	OFF	DEF	SPG	BPG	TO	PF".split()
        for row in rows[:50]:
            rank = row.find_element(By.XPATH, './td[1]').text
            dict_data["rank"].append(rank)
            name = row.find_element(By.XPATH, './td[2]//span[@class="ng-binding"]').text + " " + \
                row.find_element(By.XPATH, './td[2]//span[2]').text
            dict_data["name"].append(name)
            team = row.find_element(By.XPATH, './td[3]/a').text
            dict_data["team"].append(team)
            per_game_stats = [row.find_element(By.XPATH, f'./td[{i}]').text for i in range(4, 20)]  # Adjust indexes based on your needs
            for stat_name, stat in zip(stat_names, per_game_stats):
                dict_data[stat_name].append(stat)
    
    finally:
        # Close the driver after extraction is complete
        driver.quit()

    return pd.DataFrame(dict_data)


In [22]:
scrape_nba_data("https://global.nba.com/statistics/")

Exception managing MicrosoftEdge: error sending request for url (https://msedgedriver.azureedge.net/LATEST_RELEASE_131_WINDOWS)
Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


,rank,name,team,G,GS,PPG,RPG,APG,MPG,EFF,FG%,3P%,FT%,OFF,DEF,SPG,BPG,TO,PF
0,1,Giannis Antetokounmpo,MIL,16,16,32.4,11.9,6.4,35,37,60.8,21.4,60.2,2.1,9.8,0.6,1.4,3.3,2.9
1,2,LaMelo Ball,CHA,17,17,31,5.1,6.9,33.9,25.6,44.0,35.6,86.0,0.9,4.2,1.2,0.2,4.4,4.1
2,3,Anthony Davis,LAL,16,16,29.8,11.4,2.9,35.2,34.6,55.4,38.2,77.8,2.6,8.8,1.3,2.1,2.1,2
3,4,Nikola Jokić,DEN,13,13,29.7,13.4,10.9,37.6,42.8,55.9,52.7,83.9,4.3,9.1,1.5,0.8,3.8,1.8
4,5,Shai Gilgeous-Alexander,OKC,17,17,29.2,5.1,6.5,34.1,29.9,50.9,35.5,87.5,0.8,4.2,1.7,1.1,2.8,1.8
5,6,Paolo Banchero,ORL,5,5,29,8.8,5.6,36.4,28.6,49.5,34.4,64.4,2.4,6.4,0.6,0.8,2.2,2.6
6,7,Jayson Tatum,BOS,18,18,28.4,8.2,5.8,36.3,29.1,45.1,37.3,80.1,0.4,7.8,1.4,0.6,2.8,2.6
7,8,Luka Dončić,DAL,14,14,28.1,7.6,7.6,36.6,27.8,43.5,32.4,78.3,0.6,7.1,1.6,0.4,3.3,2.8
8,9,De'Aaron Fox,SAC,18,18,28.1,4.8,5.7,37.9,25.5,50.4,34.5,81.6,0.9,3.9,1.7,0.1,3.6,2.7
9,10,Anthony Edwards,MIN,17,17,28,5.5,3.7,37.5,23.8,46.5,42.6,80.8,0.6,4.9,1.1,0.7,3.1,2.2


### 1.2 Crawling Players Personal Information

Inspect the webpage at https://global.nba.com/playerindex/

It provides personal information of all the players in the season along with the functionality to filter the players by name.

To be able to filter the players in the webpage, you must perform the scraping using the package `Selenium`.

Create another SQL table named **players_personal_info**. this one and the table in the previous questions must be related via a defined key.

In [ ]:

def scrape_player_personal_info(url):
    dict_data = defaultdict(list)
    driver = webdriver.Edge() 
    
    try:
        driver.get(url)

        # Wait for the player index list to be present
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, '//div[@class="nba-player-index__list"]'))
        )

        # Locate the player list and extract player data
        players = driver.find_elements(By.XPATH, '//div[@class="nba-player-index__list"]/div')

        for player in players:
            name = player.find_element(By.XPATH, './/a[@class="nba-player-index__name"]').text.strip()
            team = player.find_element(By.XPATH, './/span[@class="nba-player-index__team"]').text.strip()
            position = player.find_element(By.XPATH, './/span[@class="nba-player-index__position"]').text.strip()
            height = player.find_element(By.XPATH, './/span[@class="nba-player-index__height"]').text.strip()
            weight = player.find_element(By.XPATH, './/span[@class="nba-player-index__weight"]').text.strip()

            dict_data["NAME"].append(name)
            dict_data["TEAM"].append(team)
            dict_data["POSITION"].append(position)
            dict_data["HEIGHT"].append(height)
            dict_data["WEIGHT"].append(weight)

    except Exception as e:
        print(f"An error occurred: {e}")
    
    finally:
        driver.quit()  # Ensure the driver is closed after extraction

    return pd.DataFrame(dict_data)


In [39]:
scrape_player_personal_info("https://global.nba.com/playerindex/")

Exception managing MicrosoftEdge: error sending request for url (https://msedgedriver.azureedge.net/LATEST_RELEASE_131_WINDOWS)


An error occurred: Message: 
Stacktrace:
	(No symbol) [0x00007FF7F5CF6B15]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF7F601F4A4+1437348]
	sqlite3_dbdata_init [0x00007FF7F60C2DE6+643190]
	(No symbol) [0x00007FF7F5C1C9DB]
	(No symbol) [0x00007FF7F5C1CAE3]
	(No symbol) [0x00007FF7F5C592F7]
	(No symbol) [0x00007FF7F5C3C1DF]
	(No symbol) [0x00007FF7F5C13437]
	(No symbol) [0x00007FF7F5C56BFF]
	(No symbol) [0x00007FF7F5C3BE03]
	(No symbol) [0x00007FF7F5C12984]
	(No symbol) [0x00007FF7F5C11E30]
	(No symbol) [0x00007FF7F5C12571]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF7F5FCBB34+1094964]
	(No symbol) [0x00007FF7F5D332C8]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF7F5FCAF73+1091955]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF7F5FCAAD9+1090777]
	Microsoft::Applications::Events::ILogConfiguration::operator* [0x00007FF7F5DD0CE1+461569]
	Microsoft::Applications::Events::ILogConfiguration::operator* [0x00007FF7F

""


### 1.3 Find out the cause!

Next, we'll explore the causal relationships between specific player attributes and their ranking within the top 50. Apply the techniques you learned in class, including propensity score matching, t-tests, and A-B testing, to analyze the following potential causations:

* Does having African heritage contribute to better performance in the NBA?

* Does height cause improved performance in the NBA?

## 2. Basketball for life!

In this question, we want to affirm our analyses based on a more populated dataset. Seasons statistics has been provided from 1950 to 2022. The dataset is freely avaiable at <a href="https://www.kaggle.com/datasets/blitzapurv/nba-players-data-1950-to-2021">Kaggle</a>. However, the dataset is provided at directory `./data/Q2/` as well in case of unstable internet connections.

The specfications of the columns can be view at <a href="https://www.kaggle.com/datasets/blitzapurv/nba-players-data-1950-to-2021?select=seasons_stats.csv">this wepbage</a>.

### 2.1 Cleaning Phase 

The season statistics such as field goals, 3-pointer success rates, assists, ... are of crucial importance to our analyses. Hence, drop the records with null values in these fields.

In [30]:
# Load and read the csv files
player_data_path = 'data/Q2/player_data.csv'
player_data = pd.read_csv(player_data_path)

seasons_stats_path = 'data/Q2/seasons_stats.csv'
seasons_stats = pd.read_csv(seasons_stats_path, encoding='latin1')

# Drop records with null values in crucial fields
crucial_fields = ['FG', '3P%', 'AST']
seasons_stats = seasons_stats.dropna(subset=crucial_fields)

# Display the shape
seasons_stats.shape, seasons_stats


((18632, 51),
        Unnamed: 0  Year                Player Pos   Age   Tm   G    GS  \
 5697         5727  1980  Kareem Abdul-Jabbar*   C  32.0  LAL  82   NaN   
 5698         5728  1980         Tom Abernethy  PF  25.0  GSW  67   NaN   
 5699         5729  1980           Alvan Adams   C  25.0  PHO  75   NaN   
 5700         5730  1980       Tiny Archibald*  PG  31.0  BOS  80  80.0   
 5702         5732  1980            Gus Bailey  SG  28.0  WSB  20   NaN   
 ...           ...   ...                   ...  ..   ...  ...  ..   ...   
 28052       28123  2022          Delon Wright  PG  28.0  SAC  27   8.0   
 28053       28124  2022        Thaddeus Young  PF  32.0  CHI  68  23.0   
 28054       28125  2022            Trae Young  PG  22.0  ATL  63  63.0   
 28055       28126  2022           Cody Zeller   C  28.0  CHO  48  21.0   
 28056       28127  2022           Ivica Zubac   C  23.0  LAC  72  33.0   
 
            MP   FG  ...  TOV%  USG%  OWS  DWS    WS  WS/48  OBPM  DBPM  BPM  \
 569

### 2.2 Extracting Meaningful Signals 

From the dataset, identify the most important statistical factors contributing to a player's overall performance score. Your task is to combine the provided statistics—**field goals**, **2-pointer success rate**, **3-pointer success rate**, **assists**, **blocks**, **steals**, **rebounds**, **minutes played**, and other relevant fields—into a single performance indicator.

A simple approach could be to combine the attributes together using a linear model. Feel free to use any methods to adjust the weights. Plot your performance indicator to inspect the distribution visually.

In [29]:
# List of factors
performance_factors = ['FG', '2P%', '3P%', 'AST', 'BLK', 'STL', 'TRB', 'MP']

# Dropping null values
performance_data = seasons_stats[performance_factors].dropna()

# Normalizing attributes via Min-Max scaling
normalized_data = (performance_data - performance_data.min()) / (performance_data.max() - performance_data.min())

# Assigning weights to each attribute

weights = {
    'FG': 0.2,     
    '2P%': 0.17,   
    '3P%': 0.2,   
    'AST': 0.14,   
    'BLK': 0.12,    
    'STL': 0.12,    
    'TRB': 0.12,    
    'MP': 0.05     
}

# Computing the performance indicator (PI)
performance_data['PI'] = normalized_data.apply(lambda row: sum(row[key] * weights[key] for key in weights), axis=1)

# Displaying dataset
performance_data[['PI']]


,PI
5697,0.530290
5698,0.170270
5699,0.352150
5700,0.379085
5702,0.289714
...,...
28052,0.233729
28053,0.352521
28054,0.386249
28055,0.229393


### 2.3 Hypothesis Tests

Examine the following hypothesis tests using the methods discussed in class, such as ANOVA, t-tests, A-B testing, Pearson and Spearman correlations. Make sure to provide p-values for each experiment and thoroughly justify your conclusions.

* Hypothesis 1: Player performance has significantly increased over time.

* Hypothesis 2: The average height and weight of NBA players has increased significantly over time.

* Hypothesis 3: Players from *Kentucky* college have a higher performance than players from other colleges.

* Hypothesis 4: There is a significant correlation between a player's height and their average points per game.

* Hypothesis 1: Player performance has significantly increased over time.

In [31]:
from scipy.stats import f_oneway, pearsonr, spearmanr, ttest_ind, mannwhitneyu

# Extracting years and PIs
performance_data['Year'] = seasons_stats['Year']  # Add Year column for grouping
yearly_performance = performance_data.groupby('Year')['PI'].mean().reset_index()

# ANOVA: Comparing means across groups of years (dividing into three eras for simplicity)
era1 = yearly_performance[yearly_performance['Year'] <= 1980]['PI']
era2 = yearly_performance[(yearly_performance['Year'] > 1980) & (yearly_performance['Year'] <= 2000)]['PI']
era3 = yearly_performance[yearly_performance['Year'] > 2000]['PI']

anova_result = f_oneway(era1, era2, era3)

# Correlation analysis: Pearson and Spearman between year and PI
pearson_corr, pearson_p = pearsonr(yearly_performance['Year'], yearly_performance['PI'])
spearman_corr, spearman_p = spearmanr(yearly_performance['Year'], yearly_performance['PI'])

# Mann-Whitney U Test: Comparing early and recent eras
mannwhitney_result = mannwhitneyu(era1, era3, alternative='two-sided')

# Displaying results
results = {
    "ANOVA F-statistic": anova_result.statistic,
    "ANOVA p-value": anova_result.pvalue,
    "Pearson Correlation": pearson_corr,
    "Pearson p-value": pearson_p,
    "Spearman Correlation": spearman_corr,
    "Spearman p-value": spearman_p,
    "Mann-Whitney U statistic": mannwhitney_result.statistic,
    "Mann-Whitney p-value": mannwhitney_result.pvalue
}
results


{'ANOVA F-statistic': np.float64(9.900985609634931),
 'ANOVA p-value': np.float64(0.00032128454429833933),
 'Pearson Correlation': np.float64(-0.6505441085072895),
 'Pearson p-value': np.float64(2.3316733992391964e-06),
 'Spearman Correlation': np.float64(-0.7838562319174311),
 'Spearman p-value': np.float64(5.10075397425554e-10),
 'Mann-Whitney U statistic': np.float64(22.0),
 'Mann-Whitney p-value': np.float64(0.11334723786111076)}

* Hypothesis 2: The average height and weight of NBA players has increased significantly over time.

In [32]:
# Extracting Heights and Weights from player_data and clean
player_height_weight = player_data[['Player', 'Ht', 'Wt']].dropna()

# Converting heights from "feet-inches"  to inches
player_height_weight['Ht'] = player_height_weight['Ht'].apply(
    lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1])
)

# Joining heights and weights data with seasons_stats using the "Player" column
seasons_stats_with_hw = seasons_stats.merge(
    player_height_weight, on='Player', how='inner'
)
seasons_stats_with_hw = seasons_stats_with_hw[['Year', 'Player', 'Ht', 'Wt']]

# Group by Year and calculate average height and weight
yearly_avg_hw = seasons_stats_with_hw.groupby('Year')[['Ht', 'Wt']].mean().reset_index()

# Split into eras for statistical testing
era1_height = yearly_avg_hw[yearly_avg_hw['Year'] <= 1980]['Ht']
era2_height = yearly_avg_hw[(yearly_avg_hw['Year'] > 1980) & (yearly_avg_hw['Year'] <= 2000)]['Ht']
era3_height = yearly_avg_hw[yearly_avg_hw['Year'] > 2000]['Ht']
era1_weight = yearly_avg_hw[yearly_avg_hw['Year'] <= 1980]['Wt']
era2_weight = yearly_avg_hw[(yearly_avg_hw['Year'] > 1980) & (yearly_avg_hw['Year'] <= 2000)]['Wt']
era3_weight = yearly_avg_hw[yearly_avg_hw['Year'] > 2000]['Wt']

# Performing ANOVA tests
anova_height = f_oneway(era1_height, era2_height, era3_height)
anova_weight = f_oneway(era1_weight, era2_weight, era3_weight)

# Correlation analysis for height and weight over years
pearson_height, pearson_height_p = pearsonr(yearly_avg_hw['Year'], yearly_avg_hw['Ht'])
spearman_height, spearman_height_p = spearmanr(yearly_avg_hw['Year'], yearly_avg_hw['Ht'])

pearson_weight, pearson_weight_p = pearsonr(yearly_avg_hw['Year'], yearly_avg_hw['Wt'])
spearman_weight, spearman_weight_p = spearmanr(yearly_avg_hw['Year'], yearly_avg_hw['Wt'])

# T-tests for early vs recent years
t_test_height = ttest_ind(era1_height, era3_height)
t_test_weight = ttest_ind(era1_weight, era3_weight)

# results
height_weight_results = {
    "Height ANOVA F-statistic": anova_height.statistic,
    "Height ANOVA p-value": anova_height.pvalue,
    "Weight ANOVA F-statistic": anova_weight.statistic,
    "Weight ANOVA p-value": anova_weight.pvalue,
    "Height Pearson Correlation": pearson_height,
    "Height Pearson p-value": pearson_height_p,
    "Height Spearman Correlation": spearman_height,
    "Height Spearman p-value": spearman_height_p,
    "Weight Pearson Correlation": pearson_weight,
    "Weight Pearson p-value": pearson_weight_p,
    "Weight Spearman Correlation": spearman_weight,
    "Weight Spearman p-value": spearman_weight_p,
}



height_weight_results

{'Height ANOVA F-statistic': np.float64(0.6666981587289065),
 'Height ANOVA p-value': np.float64(0.5190126965511098),
 'Weight ANOVA F-statistic': np.float64(83.7205658038575),
 'Weight ANOVA p-value': np.float64(5.050127804092222e-15),
 'Height Pearson Correlation': np.float64(-0.08208345145206111),
 'Height Pearson p-value': np.float64(0.6007788848906057),
 'Height Spearman Correlation': np.float64(-0.09906372697070372),
 'Height Spearman p-value': np.float64(0.5273738269364147),
 'Weight Pearson Correlation': np.float64(0.9663832224242281),
 'Weight Pearson p-value': np.float64(8.349023496921802e-26),
 'Weight Spearman Correlation': np.float64(0.9589247961340983),
 'Weight Spearman p-value': np.float64(4.731346090679551e-24)}

* Hypothesis 3: Players from *Kentucky* college have a higher performance than players from other colleges.

In [13]:
# Filtering kentucky players from player_data
kentucky_players = player_data[player_data['Colleges'] == 'Kentucky']['Player']

# Adding a column to identify kentucky players in seasons_stats
seasons_stats['Is_Kentucky'] = seasons_stats['Player'].isin(kentucky_players)

# List of factors for PI
performance_factors = ['FG', '2P%', '3P%', 'AST', 'BLK', 'STL', 'TRB', 'MP']

required_factors = seasons_stats[performance_factors].dropna()

# PI
normalized_factors = (required_factors - required_factors.min()) / (required_factors.max() - required_factors.min())
weights = {
    'FG': 0.2, '2P%': 0.15, '3P%': 0.18, 'AST': 0.15,
    'BLK': 0.1, 'STL': 0.1, 'TRB': 0.1, 'MP': 0.02
}
seasons_stats['PI'] = normalized_factors.apply(
    lambda row: sum(row[col] * weights[col] for col in performance_factors), axis=1
)

# Separating PI and removing null values
kentucky_pi = seasons_stats[seasons_stats['Is_Kentucky']]['PI']
non_kentucky_pi = seasons_stats[~seasons_stats['Is_Kentucky']]['PI']
kentucky_pi_clean = kentucky_pi.dropna()
non_kentucky_pi_clean = non_kentucky_pi.dropna()

# Performing t-test
t_test_result = ttest_ind(kentucky_pi_clean, non_kentucky_pi_clean, equal_var=False)

# Performing Mann-Whitney U Test
mannwhitney_result = mannwhitneyu(kentucky_pi_clean, non_kentucky_pi_clean, alternative='two-sided')

# Results
test_results = {
    "T-test statistic": t_test_result.statistic,
    "T-test p-value": t_test_result.pvalue,
    "Mann-Whitney U statistic": mannwhitney_result.statistic,
    "Mann-Whitney p-value": mannwhitney_result.pvalue
}


test_results

{'T-test statistic': np.float64(4.224476797138021),
 'T-test p-value': np.float64(2.8293670955474243e-05),
 'Mann-Whitney U statistic': np.float64(4930873.0),
 'Mann-Whitney p-value': np.float64(3.670426189736064e-05)}

* Hypothesis 4: There is a significant correlation between a player's height and their average points per game.

In [15]:
# Filtering relevant columns one more time
player_height = player_data[['Player', 'Ht']].dropna()
player_height['Ht'] = player_height['Ht'].apply(
    lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1])  # Convert height to inches
)

# Extracting relevant columns from seasons_stats
player_points = seasons_stats[['Player', 'PTS', 'G']].dropna()

# Calculating average PPG
player_points['PPG'] = player_points['PTS'] / player_points['G']

# Merging height and PPG data using the Player column
height_points_data = pd.merge(player_height, player_points[['Player', 'PPG']], on='Player', how='inner')

# Performing Pearson and Spearman correlations
pearson_corr, pearson_p = pearsonr(height_points_data['Ht'], height_points_data['PPG'])
spearman_corr, spearman_p = spearmanr(height_points_data['Ht'], height_points_data['PPG'])

# Correlation results
correlation_results = {
    "Pearson Correlation": pearson_corr,
    "Pearson p-value": pearson_p,
    "Spearman Correlation": spearman_corr,
    "Spearman p-value": spearman_p
}

correlation_results

{'Pearson Correlation': np.float64(0.05013024765920483),
 'Pearson p-value': np.float64(2.245069045589643e-11),
 'Spearman Correlation': np.float64(0.03612257998615512),
 'Spearman p-value': np.float64(1.4460087240839277e-06)}